<center><h1>XGBOOST TRAINING</h1><center>

In [5]:
# ============================================================
#  - Year cap: 2008–2024
#  - Focus genres: Pop, Hip-Hop, Latin, Electronic
#  - Target: median_track_popularity
#  - Features (edit list if your CSV uses different names)
#  - Time-aware CV: 3y train / 2y test, 1y step, **all folds**
#  - Manual baseline params per genre, then grid per fold; winner by RMSE
#  - Final params per genre: lowest average RMSE across folds
#  - Holdout: train ≤ 2022 → predict 2023–2024
#  - Saves: WF metrics, WF candidates (all tried), holdout results, feature importance, summary
# ============================================================

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

In [6]:
# -------------------
# CONFIG (edit these)
# -------------------
YEAR_MIN, YEAR_MAX = 2008, 2024
TRAIN_YEARS, TEST_YEARS = 3, 2
RANDOM_STATE = 42

# Thesis thresholds for reporting (for XGB we’ll show them but not hard-pass/fail)
THRESH_RMSE, THRESH_MAE, THRESH_R2 = 8.0, 6.0, 0.70

FOCUS_GENRES = ["pop", "hip-hop", "latin", "electronic"]

# Feature columns in your aggregated CSV (adjust if needed)
FEATURES = [
    "median_energy",
    "median_tempo",
    "median_danceability",
    "median_loudness",
    "median_liveness",
    "median_valence",
    "median_speechiness",
    "median_instrumentalness",
    "median_acousticness",
]
TARGET = "median_track_popularity"

# Manual baseline hyperparameters per genre (you can tweak per genre if you like)
MANUAL_PARAMS = {
    "pop": {
        "max_depth": 4, "learning_rate": 0.08, "n_estimators": 300,
        "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1.0, "reg_alpha": 0.0
    },
    "hip-hop": {
        "max_depth": 4, "learning_rate": 0.08, "n_estimators": 300,
        "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1.0, "reg_alpha": 0.0
    },
    "latin": {
        "max_depth": 4, "learning_rate": 0.08, "n_estimators": 300,
        "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1.0, "reg_alpha": 0.0
    },
    "electronic": {
        "max_depth": 4, "learning_rate": 0.08, "n_estimators": 300,
        "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1.0, "reg_alpha": 0.0
    },
}

# Grid for hybrid tuning (kept moderate for annual data sizes)
GRID = {
    "max_depth":         [3, 4, 5, 6],
    "learning_rate":     [0.03, 0.05, 0.08, 0.1],
    "n_estimators":      [200, 300, 400, 600],
    "subsample":         [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree":  [0.7, 0.8, 0.9, 1.0],
    "reg_lambda":        [0.0, 0.5, 1.0, 2.0],
    "reg_alpha":         [0.0, 0.1, 0.5],
}

def _build_param_grid(grid):
    # Cartesian product of grid dict -> list of dicts
    import itertools
    keys = list(grid.keys())
    vals = [grid[k] for k in keys]
    combos = []
    for v in itertools.product(*vals):
        combos.append({k: v[i] for i, k in enumerate(keys)})
    return combos

GRID_COMBOS = _build_param_grid(GRID)

# ---------------
# Helper functions
# ---------------
def _windows(years, tr=3, te=2):
    """Return **all** contiguous windows (no cap)."""
    res, total = [], tr + te
    for i in range(0, len(years) - total + 1):
        a, b = years[i:i+tr], years[i+tr:i+total]
        if all(np.diff(a) == 1) and all(np.diff(b) == 1) and a[-1] + 1 == b[0]:
            res.append((a, b))
    return res

def _metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    return rmse, mae, r2

def _fit_predict_xgb(X_train, y_train, X_test, params):
    model = XGBRegressor(
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
        **params
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return preds, model

def run_thesis_xgboost(aggregated_df, genres=FOCUS_GENRES, out_dir="outputs/xgb"):
    os.makedirs(out_dir, exist_ok=True)

    df = aggregated_df.copy()
    if "playlist_genre" not in df.columns or "track_album_release_date" not in df.columns:
        raise ValueError("Required columns missing: playlist_genre, track_album_release_date")

    # Year normalize & cap
    if np.issubdtype(df["track_album_release_date"].dtype, np.number):
        df["year"] = df["track_album_release_date"].astype(int)
    else:
        df["year"] = pd.to_datetime(df["track_album_release_date"], errors="coerce").dt.year
    df = df.dropna(subset=["year"]); df["year"] = df["year"].astype(int)
    df = df[(df["year"] >= YEAR_MIN) & (df["year"] <= YEAR_MAX)]

    # Filter genres
    df["playlist_genre"] = df["playlist_genre"].str.lower()
    df = df[df["playlist_genre"].isin([g.lower() for g in genres])]
    df = df.sort_values(["playlist_genre", "year"])

    # Check features & target
    missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
    if missing:
        raise ValueError(f"Aggregated CSV is missing columns: {missing}")

    # Log grid used (appendix)
    with open(os.path.join(out_dir, "grid_xgb_used.txt"), "w") as f:
        f.write("Manual params per genre:\n")
        for g in genres:
            f.write(f"{g}: {MANUAL_PARAMS.get(g, {})}\n")
        f.write("\nGrid:\n")
        for p in GRID_COMBOS:
            f.write(f"{p}\n")

    summary_rows = []

    for genre in genres:
        g = df[df["playlist_genre"] == genre].copy()
        if g.empty:
            continue

        # Build per-year design matrix
        g = g.set_index("year").sort_index()
        years_all = g.index.tolist()
        if len(years_all) < 5:
            print(f"[{genre}] Not enough history — skipping.")
            continue

        X_all = g[FEATURES].astype(float)
        y_all = g[TARGET].astype(float)

        # Holdout split
        holdout_years = [y for y in (2023, 2024) if y in years_all]
        years_wf = [y for y in years_all if y not in holdout_years]

        # Windows: **all** 3y train / 2y test
        windows = _windows(years_wf, TRAIN_YEARS, TEST_YEARS)
        if len(windows) < 10:
            print(f"[{genre}] Only {len(windows)} folds available.")

        fold_rows = []
        candidate_rows = []   # all candidates per fold
        param_avg_rmse = {}   # params(str) -> (sum_rmse, count)
        final_best_params = None

        # Walk-forward
        for fold_id, (tr, te) in enumerate(windows, start=1):
            X_tr, y_tr = X_all.loc[tr], y_all.loc[tr]
            X_te, y_te = X_all.loc[te], y_all.loc[te]

            # 1) Manual baseline
            manual = MANUAL_PARAMS.get(genre, MANUAL_PARAMS["pop"])
            preds, _model = _fit_predict_xgb(X_tr, y_tr, X_te, manual)
            best_rmse, best_mae, best_r2 = _metrics(y_te, preds)
            best_params = manual.copy()

            candidate_rows.append({
                "genre": genre, "fold": fold_id,
                "train_start": tr[0], "train_end": tr[-1],
                "test_start": te[0],  "test_end": te[-1],
                **{f"param_{k}": v for k, v in manual.items()},
                "rmse": best_rmse, "mae": best_mae, "r2": best_r2,
                "source": "manual"
            })

            # 2) Grid tuning per fold
            for p in GRID_COMBOS:
                try:
                    preds_g, _mdl = _fit_predict_xgb(X_tr, y_tr, X_te, p)
                    rmse_g, mae_g, r2_g = _metrics(y_te, preds_g)
                    candidate_rows.append({
                        "genre": genre, "fold": fold_id,
                        "train_start": tr[0], "train_end": tr[-1],
                        "test_start": te[0],  "test_end": te[-1],
                        **{f"param_{k}": v for k, v in p.items()},
                        "rmse": rmse_g, "mae": mae_g, "r2": r2_g,
                        "source": "grid"
                    })
                    if rmse_g < best_rmse:
                        best_rmse, best_mae, best_r2 = rmse_g, mae_g, r2_g
                        best_params = p.copy()
                except Exception:
                    continue

            # Record fold winner
            fold_rows.append({
                "genre": genre, "fold": fold_id,
                "train_start": tr[0], "train_end": tr[-1],
                "test_start": te[0],  "test_end": te[-1],
                "rmse": best_rmse, "mae": best_mae, "r2": best_r2,
                **{f"best_{k}": v for k, v in best_params.items()}
            })
            key = str(sorted(best_params.items()))
            s, c = param_avg_rmse.get(key, (0.0, 0))
            param_avg_rmse[key] = (s + best_rmse, c + 1)

        # Save per-genre fold winners & all candidates
        wf_df = pd.DataFrame(fold_rows)
        wf_df.to_csv(os.path.join(out_dir, f"{genre}_xgb_walkforward_metrics.csv"), index=False)
        cand_df = pd.DataFrame(candidate_rows)
        cand_df.to_csv(os.path.join(out_dir, f"{genre}_xgb_wf_candidates.csv"), index=False)

        # Choose final params by lowest average RMSE across folds
        if not cand_df.empty:
            # Extract params columns
            param_cols = [c for c in cand_df.columns if c.startswith("param_")]
            # Build a key for each unique set of params
            cand_df["params_key"] = cand_df[param_cols].apply(
                lambda r: tuple((c.replace("param_", ""), r[c]) for c in param_cols),
                axis=1
            )
            # Group by params_key and compute mean RMSE
            best_group = cand_df.groupby("params_key")["rmse"].mean().idxmin()
            final_params = dict(best_group)
        else:
            # Fallback to manual if no candidates (unlikely)
            final_params = MANUAL_PARAMS.get(genre, MANUAL_PARAMS["pop"])

        # Holdout 2023–2024 using final_params
        hold_rmse = hold_mae = hold_r2 = np.nan
        if set((2023, 2024)).issubset(set(years_all)):
            pre = [y for y in years_all if y <= 2022]
            if len(pre) >= 3:
                X_tr_full, y_tr_full = X_all.loc[pre], y_all.loc[pre]
                X_hold, y_hold = X_all.loc[[2023, 2024]], y_all.loc[[2023, 2024]]
                y_pred_hold, _ = _fit_predict_xgb(X_tr_full, y_tr_full, X_hold, final_params)
                hold_rmse, hold_mae, hold_r2 = _metrics(y_hold, y_pred_hold)
                pd.DataFrame({"year":[2023, 2024], "actual": y_hold.values, "forecast": y_pred_hold}).to_csv(
                    os.path.join(out_dir, f"{genre}_xgb_holdout_2023_2024_vs_forecast.csv"), index=False
                )

        # Fit final model on full capped series, save feature importances
        final_model = XGBRegressor(random_state=RANDOM_STATE, objective="reg:squarederror", **final_params)
        final_model.fit(X_all, y_all)
        importances = final_model.get_booster().get_score(importance_type="gain")
        
        # Normalize & save
        imp_df = pd.DataFrame({
            "feature": list(importances.keys()),
            "gain_importance": list(importances.values())
        }).sort_values("gain_importance", ascending=False)
        # Map XGBoost feature names (f0, f1, ...) back to original column names if needed
        # XGBRegressor preserves names if using pandas, but fallback mapping:
        if imp_df["feature"].str.startswith("f").all():
            # build index mapping
            feat_map = {f"f{i}": FEATURES[i] for i in range(len(FEATURES))}
            imp_df["feature"] = imp_df["feature"].map(feat_map)
        imp_df.to_csv(os.path.join(out_dir, f"{genre}_xgb_feature_importance.csv"), index=False)

        # Plot top 10 importances
        plt.figure(figsize=(7,4))
        top = imp_df.head(10).iloc[::-1]
        plt.barh(top["feature"], top["gain_importance"])
        plt.title(f"{genre.title()} — XGBoost Feature Importance (gain)")
        plt.xlabel("Gain"); plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{genre}_xgb_feature_importance.png"))
        plt.close()

        # Walk-forward averages for reporting
        avg_rmse = float(wf_df["rmse"].mean()) if not wf_df.empty else np.nan
        avg_mae  = float(wf_df["mae"].mean())  if not wf_df.empty else np.nan
        avg_r2   = float(wf_df["r2"].mean())   if not wf_df.empty else np.nan

        summary_rows.append({
            "genre": genre,
            "folds": int(len(windows)),
            "avg_rmse_walkforward": avg_rmse,
            "avg_mae_walkforward":  avg_mae,
            "avg_r2_walkforward":   avg_r2,
            "final_params": str(final_params),
            "holdout_rmse_2023_2024": hold_rmse,
            "holdout_mae_2023_2024":  hold_mae,
            "holdout_r2_2023_2024":   hold_r2,
            "meets_thresholds_walkforward": bool(
                (avg_rmse <= THRESH_RMSE) and (avg_mae <= THRESH_MAE) and (avg_r2 >= THRESH_R2)
            ) if not (np.isnan(avg_rmse) or np.isnan(avg_mae) or np.isnan(avg_r2)) else False
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(out_dir, "xgb_summary_by_genre.csv"), index=False)
    print("Done. Outputs in:", out_dir)
    return summary_df

In [7]:
df = pd.read_csv("05_genres_median_audio_features.csv")
xgb_summary = run_thesis_xgboost(df, genres=['pop','hip-hop','latin','electronic'], out_dir="outputs/xgb")
xgb_summary

TypeError: 'numpy.float64' object cannot be interpreted as an integer